# Etapa 3 — Modelos

**Sumário do trabalho**

1. **Análise exploratória** (`01_analise_exploratoria.ipynb`): base, faltantes, alvo e análise bivariada.
2. **Tratamento dos dados** (`02_tratamento_dados.ipynb`): remoção de vazamento; faltantes (estrutural, exame não pedido, esporádico); codificação das qualitativas; remoção de colunas vazias, constantes e redundantes.
3. **Modelos** (`03_modelos.ipynb`): divisão treino/teste 80/20; árvore de decisão podada, regressão logística Lasso (seleciona os atributos) e Random Forest; hiperparâmetros por validação cruzada de 10 *folds*. ← *este notebook*
4. **Avaliação** (`04_avaliacao.ipynb`): AUC na validação cruzada e no teste, com intervalo de confiança e comparação com o acaso; matriz de confusão; uso clínico (limiar para não perder eventos e curva de decisão); interpretação dos três modelos.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src import utils, preprocessing, features, train

utils.setup_plotting()

### Divisão treino/teste

80/20, estratificada pelo alvo. O teste só é usado na etapa 4.

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()
variable_groups = preprocessing.get_variable_groups(patients)
X, y, _ = features.build_dataset(patients, drugs, variable_groups)

X_train, X_test, y_train, y_test = preprocessing.split_train_test(X, y)

pd.DataFrame({
    "internações": [len(y_train), len(y_test)],
    "eventos": [int(y_train.sum()), int(y_test.sum())],
    "eventos_pct": [round(y_train.mean() * 100, 1), round(y_test.mean() * 100, 1)],
}, index=["treino", "teste"])

### Modelos

| Modelo | Pipeline | Hiperparâmetros |
|---|---|---|
| Árvore de decisão | mediana → árvore | poda por complexidade de custo (`ccp_alpha`) |
| Regressão logística Lasso | mediana → padronização → logística L1 | penalidade (`C`); zera os atributos que não ajudam |
| Random Forest | mediana → 500 árvores com *bagging* | atributos sorteados por divisão (`max_features`), tamanho mínimo da folha |

Os três usam pesos por classe (`class_weight="balanced"`) para compensar o desbalanceamento.

### Validação cruzada

Busca em grade com validação cruzada estratificada de 10 *folds* no treino, métrica AUC. `auc_cv` é a média nos 10 *folds* de validação; `auc_treino`, nos próprios dados de treino (a distância entre as duas indica sobreajuste).

In [ ]:
cv_results, searches = train.tune_models(X_train, y_train)

cv_results.to_csv(utils.RESULTS_DIR / "model_cv_results.csv")
pd.set_option("display.max_colwidth", 80)
cv_results

### Tamanho dos modelos ajustados

In [ ]:
train.model_sizes(searches, X_train.shape[1]).to_frame()

### Efeito da poda na árvore

AUC de validação para cada `ccp_alpha` (0 = árvore sem poda).

In [ ]:
tree_search = searches["Árvore de decisão"]
pd.DataFrame({"ccp_alpha": tree_search.cv_results_["param_model__ccp_alpha"],
              "auc_cv": tree_search.cv_results_["mean_test_score"].round(3)})

### Os indicadores de exame não pedido ajudam?

Mesmo Lasso, mesma grade e mesmos *folds*, com e sem os 15 indicadores (os valores medidos ficam nos dois casos, imputados pela mediana).

In [ ]:
ablation = train.indicator_ablation(X_train, y_train)
ablation.to_csv(utils.RESULTS_DIR / "lasso_indicator_ablation.csv", index_label="matriz")
ablation